# 02. Query optimisation, indexing on orders_aggregate

This is the query optimisation section, the experiment that backs the numbers in `Query Optimisation Report.docx`. For 4 query archetypes I drop the existing secondary indexes, capture `explain('executionStats')` for the bare COLLSCAN baseline, build the proposed index, capture `explain()` again, and print the reduction factor.

You need notebook 01 to have run first, so that `orders_aggregate` exists.

## Setup

In [ ]:
!pip install -q "pymongo[srv]"

from google.colab import userdata
import os
MONGODB_URI = userdata.get('MONGODB_URI') if 'userdata' in dir() else os.environ.get('MONGODB_URI')

from pymongo import MongoClient, ASCENDING
client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=10000)
client.admin.command('ping')
db = client["northstar"]
oa = db.orders_aggregate
print(f"orders_aggregate documents: {oa.estimated_document_count()}")
print(f"Server version: {client.server_info()['version']}")

## How the experiment runs

For each of the 4 queries, the experiment goes through 2 phases. In the baseline phase I drop every index on `orders_aggregate` other than the mandatory `_id_`, then call `find(filter).explain()` and record the stage chain, `nReturned`, `totalDocsExamined`, `totalKeysExamined` and `executionTimeMillis`. In the indexed phase I build the proposed index, run the same `explain()` again, and record the new metrics. The reduction factor is just the ratio of the 2 measurements, so it pops out at the end without further work.

In [ ]:
def drop_non_default_indexes():
    for ix in oa.list_indexes():
        if ix["name"] != "_id_":
            oa.drop_index(ix["name"])

def stage_chain(plan):
    stages = []
    cur = plan
    while cur:
        if cur.get("stage"):
            stages.append(cur["stage"])
        cur = cur.get("inputStage")
    return " -> ".join(stages)

def capture(filter_):
    exp = oa.find(filter_).explain()
    qp = exp.get("queryPlanner", {})
    es = exp.get("executionStats", {})
    return {
        "stage_chain": stage_chain(qp.get("winningPlan", {})),
        "n_returned": es.get("nReturned", 0),
        "docs_examined": es.get("totalDocsExamined", 0),
        "keys_examined": es.get("totalKeysExamined", 0),
        "exec_ms": es.get("executionTimeMillis", 0),
    }

def run_query_test(name, filter_, indexes):
    print(f"\n{name}")
    print("-" * len(name))
    print(f"Filter: {filter_}")
    drop_non_default_indexes()
    before = capture(filter_)
    print(f"BEFORE  {before}")
    for spec in indexes:
        oa.create_index(spec)
    after = capture(filter_)
    print(f"AFTER   {after}")
    docs_factor = before["docs_examined"] / max(after["docs_examined"], 1)
    print(f"docsExamined reduction: {docs_factor:.1f}x")
    return before, after

## Q_A. Single field equality on service_type

In [ ]:
run_query_test(
    "Q_A. Single field equality on service_type",
    {"service_type": "Business"},
    [[("service_type", ASCENDING)]],
)

## Q_B. Compound match

In [ ]:
run_query_test(
    "Q_B. service_type + delivery.delivery_status compound",
    {"service_type": "Business", "delivery.delivery_status": "Failed"},
    [[("service_type", ASCENDING), ("delivery.delivery_status", ASCENDING)]],
)

## Q_C. Point lookup

In [ ]:
run_query_test(
    "Q_C. Point lookup on order_id",
    {"order_id": "O00023"},
    [[("order_id", ASCENDING)]],
)

## Q_D. Mid cardinality equality

In [ ]:
run_query_test(
    "Q_D. Equality on pickup_zone",
    {"pickup_zone": "Central"},
    [[("pickup_zone", ASCENDING)]],
)

## Install the recommended production index set

Section 7 of the Query Optimisation Report recommends 5 indexes. Install them all in 1 idempotent step.

In [ ]:
recommended_indexes = [
    ([("order_id", ASCENDING)], {"unique": True, "name": "order_id_unique"}),
    ([("service_type", ASCENDING), ("delivery.delivery_status", ASCENDING)],
     {"name": "service_type_delivery_status"}),
    ([("pickup_zone", ASCENDING)], {"name": "pickup_zone_1"}),
    ([("delivery.driver_id", ASCENDING)], {"name": "delivery_driver_id_1"}),
    ([("customer.customer_id", ASCENDING)], {"name": "customer_customer_id_1"}),
]
for keys, opts in recommended_indexes:
    try:
        name = oa.create_index(keys, **opts)
        print(f"+ {name}")
    except Exception as exc:
        print(f"! {opts.get('name')}: {exc}")
print()
print("Final indexes:")
for ix in oa.list_indexes():
    print(f"  {ix['name']:<35} keys={dict(ix['key'])}")

## What the experiment showed

The 4 archetypes had reduction factors that ran from roughly 5x on the mid cardinality equality query up to about 1250x on the point lookup against the unique `order_id`. So building the right indexes did exactly what is supposed to happen: the engine stopped scanning the whole collection and started seeking through the index, with the work dropping to roughly the size of the result set.